<a href="https://colab.research.google.com/github/Jef-H/supervised_fields/blob/develop/Supervised_EDA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Supervised Field Detection

Alright so for this supervised learning module we're going to train the model to classify drawings as field or not-field

Two datasets for this project the main dataset is from kaggle https://www.kaggle.com/datasets/mahmoudreda55/satellite-image-classification/data


The field dataset was created via taking 125 screenshots of corn fields by hand from https://earthexplorer.usgs.gov/ and then fragmenting them into images comprable to the forest size sampling we have in the kaggle dataset.




In [32]:
import os
import requests
import shutil

def download_png_files_from_github(github_url, output_dir):
    """
    Downloads all PNG files from a specified GitHub repository directory,
    clearing the destination directory before downloading.

    Args:
        github_url (str): The GitHub URL of the repository directory.
        output_dir (str): Directory to save the downloaded PNG files.
    """
    # Ensure the output directory exists and clean it
    if os.path.exists(output_dir):
        shutil.rmtree(output_dir)  # Remove all files in the directory
    os.makedirs(output_dir, exist_ok=True)
    print(f"Cleaned and prepared output directory: {output_dir}")

    # Extract the repository name and directory from the GitHub URL
    repo_url = github_url.split("https://github.com/")[1]
    repo_name, tree_path = repo_url.split("/tree/")

    # Correctly format the path for the API (removing 'tree' and using branch name)
    branch_name = tree_path.split('/')[0]  # Assuming branch name is the first part of the path
    dir_path = '/'.join(tree_path.split('/')[1:])  # Directory path is the rest

    # GitHub API URL to list the contents of the directory
    api_url = f"https://api.github.com/repos/{repo_name}/contents/{dir_path}?ref={branch_name}"

    # Send a request to the GitHub API
    response = requests.get(api_url)
    if response.status_code != 200:
        print(f"Failed to fetch contents from {api_url}. Status code: {response.status_code}")
        return

    files = response.json()
    png_count = 0  # Counter for PNG files

    # Iterate through each file in the directory
    for file in files:
        # Check if the file is a PNG
        if file['name'].endswith('.png'):
            file_url = file['download_url']
            file_name = file['name']
            file_path = os.path.join(output_dir, file_name)

            # Download and save the PNG file
            file_response = requests.get(file_url)
            if file_response.status_code == 200:
                with open(file_path, 'wb') as f:
                    f.write(file_response.content)
                png_count += 1  # Increment the PNG counter
            else:
                print(f"Failed to download {file_name}")

    print(f"Downloaded {png_count} PNG files to {output_dir}")

# Example usage
github_url = "https://github.com/Jef-H/supervised_fields/tree/develop/crop_field_images"
output_directory = "downloaded_png_files/"

download_png_files_from_github(github_url, output_directory)


Cleaned and prepared output directory: downloaded_png_files/
Downloaded 125 PNG files to downloaded_png_files/


In [33]:
import os
import time  # Import time module for timing
from PIL import Image
import tempfile
import shutil

def cut_images_to_fragments(input_dir, output_dir, fragment_size=(64, 64)):
    """
    Cuts all PNG images in the input directory into fragments of the given size
    and saves them to the output directory. Excess parts of the images are discarded.

    Args:
        input_dir (str): Directory containing PNG images.
        output_dir (str): Directory to save the image fragments.
        fragment_size (tuple): Size of each fragment (width, height).
    """
    start_time = time.time()  # Start the timer

    # Clear the output directory before writing new fragments
    if os.path.exists(output_dir):
        shutil.rmtree(output_dir)  # Remove the existing directory and its contents
    os.makedirs(output_dir, exist_ok=True)  # Create a fresh directory
    print(f"Ensured output directory is cleared and exists: {output_dir}")

    total_fragment_count = 0  # Counter for all fragments across all images

    # Iterate through each file in the input directory
    for filename in os.listdir(input_dir):
        if filename.endswith('.png'):
            file_path = os.path.join(input_dir, filename)
            with Image.open(file_path) as img:
                img_width, img_height = img.size

                # Calculate the number of fragments in each dimension
                num_fragments_x = img_width // fragment_size[0]
                num_fragments_y = img_height // fragment_size[1]

                total_fragments = 0

                # Generate and save each fragment
                for i in range(num_fragments_x):
                    for j in range(num_fragments_y):
                        left = i * fragment_size[0]
                        upper = j * fragment_size[1]
                        right = left + fragment_size[0]
                        lower = upper + fragment_size[1]

                        fragment = img.crop((left, upper, right, lower))

                        fragment_filename = f"{os.path.splitext(filename)[0]}_{i}_{j}.png"
                        fragment_path = os.path.join(output_dir, fragment_filename)
                        fragment.save(fragment_path)

                        total_fragments += 1
                        total_fragment_count += 1

    print(f"Total fragments created for all images: {total_fragment_count}")

    # End the timer and print the elapsed time
    elapsed_time = time.time() - start_time
    print(f"Processing completed in {elapsed_time:.2f} seconds.")

# Example usage
github_url = "https://github.com/Jef-H/supervised_fields/tree/develop/crop_field_images"
output_directory = "crop_fragments/"

# Create temporary directory outside the 'with' context to avoid deletion
temp_dir = tempfile.mkdtemp()
print(f"Temporary directory created: {temp_dir}")
input_directory = "downloaded_png_files/"

cut_images_to_fragments(input_directory, output_directory)


Temporary directory created: /tmp/tmpgvvturib
Ensured output directory is cleared and exists: crop_fragments/
Total fragments created for all images: 5343
Processing completed in 10.79 seconds.
